# 04 - The trade-off map: is a mitigation better than just degrading the model?

Every bias-mitigation method trades some accuracy for some fairness. But is the trade *worth it*?
The FAIREA idea answers this by building a **baseline**: take the original model and randomly
overwrite a growing share of its predictions. Predictions get less accurate *and* less biased, so
the baseline is the "price" of fairness you would pay for free, with no mitigation at all. A
mitigation method is only interesting if it beats that price.

**What you'll learn**
- how to collect (balanced accuracy, bias) points for several mitigation methods with `BMGridSearch`;
- how the mutation baseline is built and why it is the yardstick;
- how to classify each method into a region (win-win, good trade-off, bad trade-off, lose-lose, inverted);
- how to read the *area* a good trade-off gains over the baseline.

**Prerequisites:** notebooks 00 (data), 01 (metrics, the composite fairness score) and 03 (mitigation grid search).
**Runtime:** about 1 minute. **Data:** diabetes readmission, protected attribute `race`.

Throughout, the fairness score is `fair_score`: **0 = fair, higher = more biased**. We call it *bias*,
and both axes of the map are "lower is better".

In [1]:
from fhg_examples import setup; setup()

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from callmefair import BMGridSearch, BMInterface, BMType
from fhg_examples.datasets import SPECS, load
from callmefair.tradeoff import (baseline_curve, classify_region, compute_area,
                                 create_baseline, normalize, tradeoff_regions)
from callmefair.viz.tradeoff_map import plot_tradeoff_map

## 1. Data: a stratified 20k sample

The full diabetes table has ~99k rows. To keep this notebook fast we draw a **stratified** sample of
20,000 rows (stratified on label and race, fixed seed), so group sizes and the readmission rate are preserved.

In [3]:
spec = SPECS["diabetes"]
privileged, unprivileged = spec.attribute("race").groups     # race 1 = Caucasian (reference)

full = load("diabetes")
df, _ = train_test_split(full, train_size=20_000, random_state=0,
                         stratify=full[["readmitted", "race"]])
print(f"{len(df):,} rows | readmitted rate {df.readmitted.mean():.3f} | "
      f"Caucasian share {df.race.mean():.3f}")

20,000 rows | readmitted rate 0.112 | Caucasian share 0.765


## 2. Method points: a compact grid search

We train one logistic regression and evaluate seven mitigations (pre-, post-processing and two
pre+post chains). `BMGridSearch` returns one row per combination, with the unmitigated model first
(`baseline`). One split is noisy, so we repeat on five splits (seeds 0-4) and average; the baseline
below uses the same five seeds.

In [4]:
SEEDS = range(5)
BM_LIST = [[BMType.preReweighing], [BMType.preDisparate], [BMType.posEqqOds],
           [BMType.posCalibrated], [BMType.posROC],
           [BMType.preReweighing, BMType.posROC], [BMType.preDisparate, BMType.posROC]]

def run_grid(seed):
    bm_i = BMInterface(df, label="readmitted", protected=["race"], random_state=seed)
    grid = BMGridSearch(bm_i, LogisticRegression(max_iter=1000), BM_LIST, privileged, unprivileged)
    return pd.DataFrame(grid.run_single_sensitive(results_dir=None))

grid = pd.concat([run_grid(s) for s in SEEDS])

In [5]:
points = (grid.groupby("BM", sort=False)[["balanced_acc", "fair_score"]].mean()
          .rename(index={"baseline": "no mitigation"}))
points

,balanced_acc,fair_score
BM,,
no mitigation,0.586,0.206
preReweighing,0.584,0.221
preDisparate,0.586,0.255
posEqqOds,0.571,0.228
posCalibrated,0.504,0.507
posROC,0.589,0.337
preReweighing posROC,0.591,0.210
preDisparate posROC,0.588,0.338


Each row is now one *point* on the map: (balanced accuracy, bias). Some methods change little
(`posROC`, `preDisparate` leave the model almost untouched on this attribute); others move far.

## 3. The baseline: degrade the model on purpose

`create_baseline` trains the same model on the same splits, then overwrites a fraction `degree`
(0 to 1) of its test predictions with a constant label, and records accuracy and bias at each degree.
It takes a *factory* (`lambda: LogisticRegression()`), so every split gets a fresh model.

Two strategies are built: overwrite with `to_0` or with `to_1`. Overwriting with 0 ends in a model
that never predicts the favorable class, where the disparate impact is undefined and the composite
score jumps to its worst value. We therefore use **`to_1`**, whose bias falls smoothly.

In [6]:
baseline = create_baseline(lambda: LogisticRegression(max_iter=1000), df, "readmitted", "race",
                           privileged, unprivileged, data_splits=len(SEEDS), repetitions=5, seed=0)
curve = baseline_curve(baseline, "to_1")
curve.round(3).iloc[::2]            # every second degree

,strategy,degree,balanced_acc,fair_score
0,to_1,0.0,0.586,0.206
2,to_1,0.2,0.568,0.215
4,to_1,0.4,0.546,0.163
6,to_1,0.6,0.533,0.149
8,to_1,0.8,0.518,0.101
10,to_1,1.0,0.500,0.006


In [7]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(curve.fair_score, curve.balanced_acc, "o-", color="tab:blue", label="baseline (degrade the model)")
for d, x, y in zip(curve.degree, curve.fair_score, curve.balanced_acc):
    if d in (0, 0.5, 1):
        ax.annotate(f"{d:.0%} mutated", (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.scatter(points.fair_score, points.balanced_acc, marker="x", color="tab:red", zorder=3, label="mitigation")
for name, r in points.iterrows():
    ax.annotate(name, (r.fair_score, r.balanced_acc), textcoords="offset points", xytext=(4, -10), fontsize=7)
ax.set(xlabel="bias (fair_score, lower = fairer)", ylabel="balanced accuracy",
       title="Mitigation points vs. the degradation baseline")
ax.legend(fontsize=8); plt.show()

<Figure size 650x420 with 1 Axes>

**Reading the figure.** The blue curve starts at the unmutated model (top right: accurate but biased) and slides
to the bottom left (a coin flip: 0.5 accuracy, no bias). A point *above or to the left of the curve* buys more
fairness per lost accuracy point than random degradation does.

## 4. Normalize and classify

Accuracy and bias live on different scales, so `normalize` rescales both axes to [0, 1] using one shared
range per axis. Accuracy is turned into **loss** (0 = most accurate), so both axes are "lower is better"
and the ideal corner is the origin.

The unmutated model **O** is the first point of the baseline. Relative to O and the curve, a method falls in one region:

| Region (paper name) | Meaning |
|---|---|
| `win-win` (Win-Win) | no more biased *and* no less accurate than O |
| `good trade-off` (Good) | fairer but less accurate, and better than the baseline curve |
| `bad trade-off` (Poor) | fairer but less accurate, and *worse* than the baseline curve |
| `lose-lose` (Lose-Lose) | more biased and less accurate than O |
| `inverted` (Inverted) | more biased but more accurate than O |

In [8]:
loss, bias, methods_norm = normalize(curve.balanced_acc, curve.fair_score,
                                     dict(zip(points.index, points.itertuples(index=False))))
line = LineString(zip(bias, loss))            # x = bias, y = accuracy loss
regions = classify_region(line, methods_norm)
areas = {m: compute_area(line, p) for m, p in methods_norm.items()}

In [9]:
summary = pd.DataFrame({"region": regions, "area": areas}).round(3)
summary.join(points.round(3))

,region,area,balanced_acc,fair_score
no mitigation,win-win,0.0,0.586,0.206
preReweighing,lose-lose,0.0,0.584,0.221
preDisparate,inverted,0.0,0.586,0.255
posEqqOds,lose-lose,0.0,0.571,0.228
posCalibrated,lose-lose,0.0,0.504,0.507
posROC,inverted,0.0,0.589,0.337
preReweighing posROC,inverted,0.0,0.591,0.210
preDisparate posROC,inverted,0.0,0.588,0.338


## 5. The regions map

Regions are shaded and named in the plot; points are coloured by model. For good trade-offs the shaded patch is the **area**: the region between the
point and the baseline, bounded by the horizontal and vertical lines through the point. A larger area means a bigger
fairness-for-accuracy bargain compared with degrading the model.

In [10]:
regions_df = tradeoff_regions(baseline, grid.to_dict("records"))   # same result as the steps above
plot_tradeoff_map(baseline, regions_df)
plt.show()

<Figure size 740x520 with 1 Axes>

## 6. Interpretation

**Is this mitigation better than just degrading the model?** On this sample, for `race`: **no.**

- The unmitigated model is the reference (`win-win` with itself). Its bias is already moderate (about 0.2), so there is little to gain.
- `preReweighing`, `posEqqOds` and `posCalibrated` end up `lose-lose`: both less accurate and not fairer (`posCalibrated` collapses to near-chance accuracy).
- `posROC` and `preDisparate` are `inverted`: accuracy is kept but bias rises. Their `area` is 0 because only good trade-offs have an area.
- No method is a `good trade-off`, so there is no area to compare. Overwriting 80% of predictions at random would cut bias to about 0.1 at 0.52 accuracy, which no method matches.

A map like this protects you from a common mistake: a method that "does something" is not necessarily better than noise. Try the notebook with `gender` or `age` (swap the attribute name): regions differ per attribute and model. With a 20k sample and 5 seeds some noise remains, so treat points near the curve as ties.

## Summary

- The **baseline** (mutate predictions) is the free-lunch yardstick: any mitigation must beat it.
- `normalize` puts accuracy loss and bias on shared [0, 1] axes; `classify_region` names the region; `compute_area` scores good trade-offs.
- Both baseline and methods use `fair_score` (lower = fairer), so the orientation is the same. A single split is noisy, so average over seeds.
- `results_dir=None` keeps `BMGridSearch` from writing CSVs; we work with the returned rows.

**Next:** [05_advanced_models_and_hardware.ipynb](05_advanced_models_and_hardware.ipynb) shows other model families, custom models, GPU settings and intersectional attributes.